# CLEAR concept audit - Vietnam TUNE only

**Diagnostic. It cannot improve and must not be reported as improving any**
**screening metric.** The reported Vietnam screening AUROC is **0.7242** and this
run does not touch it.

| | |
|---|---|
| data | **Vietnam TUNE only**, 1,352 cases, by allow-list |
| locked | Vietnam REPORT and TBX11K `internal_test` - never attached, never read |
| data licence | CC0-1.0 (Dryad API, `doi:10.5061/dryad.5tb2rbpcv`). CC0 waives copyright, **not** ethics, privacy or platform obligations |
| model | CLEAR, Apache-2.0, pinned revision `142f169c8b00` |
| concepts | **9 exact** vocabulary entries. cavity, nodule and reticular opacity are **excluded**, not approximated |
| internet | **required** - the pinned weights and the DINOv2 architecture are fetched at run time |

Settings before you start: **Accelerator = GPU**, **Internet = on**,
**Persistence = Variables and Files**.

Run the cells in order. Each gate prints a verdict or raises. A raised guard is
the package working, not a bug to route around.


## 1 - Attach and import

Two private datasets must be attached: the **code** dataset (this package's
`src/`, `configs/`, `requirements.txt`) and the **data** dataset built by
`stage_kaggle_data.py`, which contains TUNE material only.


In [ ]:
import json, sys, time
from pathlib import Path
INP, WORK = Path('/kaggle/input'), Path('/kaggle/working')
# Searched recursively: Kaggle nests dataset contents differently depending on
# how they were uploaded, and a fixed depth is the wrong thing to depend on.
# More than one complete copy is an error, not something to settle by sort
# order: that is how a stale copy got used once already.
CANDS = sorted({q.parent for q in INP.rglob('concept_audit.py')})
GOOD = [c for c in CANDS if (c/'paths.py').is_file()
        and (c/'preflight.py').is_file()]
if not GOOD:
    tree = [str(q.relative_to(INP)) for q in sorted(INP.rglob('*'))[:40]]
    raise RuntimeError('The code dataset is not attached, or is incomplete.\n'
                       'What IS attached:\n  ' + '\n  '.join(tree or ['nothing']))
if len(GOOD) > 1:
    import hashlib
    raise RuntimeError('%d copies of the code package are attached; detach all '
                       'but one:\n  ' % len(GOOD) + '\n  '.join(
        '%s  %s' % (hashlib.sha256((c/'preflight.py').read_bytes()
                                   ).hexdigest()[:16], c) for c in GOOD))
SRC = GOOD[0] / 'concept_audit.py'
sys.path.insert(0, str(GOOD[0]))
import paths as P, preflight as F, concept_audit as A
print('code package :', GOOD[0])
print('pinned model :', A.HF_REPO, A.HF_REV)
# CLEAR's own loader is used, not a reimplementation: it applies the visual
# projection and the CXR preprocessing, both of which a hand-rolled encoder
# got wrong. --no-deps, or CLEAR's pins reinstall torch.
try:
    import clear as _c
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'git+https://github.com/peterhan91/CLEAR.git@'
                    + A.CLEAR_COMMIT], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy',
                    'regex'], check=True)
    import clear as _c
print('clear package:', Path(_c.__file__).parent)


## 2 - Preflight

Reports the runtime, the installed versions against the pinned ones, the resolved
model revision and asset checksums, the approved cohort and the expected output
shape, and whether any locked partition is visible. **It stops the notebook before
any download or GPU work if something is wrong.** It never relaxes a guard, changes
the input resolution or substitutes an architecture to make the run possible.


In [ ]:
rep = F.run(check_weights=True)
print(json.dumps(rep, indent=1, default=str)[:4000])
(WORK/'tables').mkdir(parents=True, exist_ok=True)
json.dump(rep, open(WORK/'tables/preflight_kaggle.json','w'), indent=1,
          default=str)
if rep['verdict'] != 'GO':
    raise RuntimeError('PREFLIGHT STOP\n- ' + '\n- '.join(rep['blockers']))
print('\npreflight: GO')


## 3 - Cohort and partition gate

Loads the allow-listed cohort and fails closed on a REPORT row in the staged
upload, a missing or duplicated id, a count that disagrees with the manifest, a
cohort fingerprint that is not the pinned one, or a missing approved image.


In [ ]:
p = P.resolve()
print(P.describe(p)); print()
d = A.load_tune_only(p)
m, primary = A.concept_map()
names = [c['concept'] for c in primary]
missing, extra = A.check_images(p, d['pid'])
print('cases              :', len(d['pid']))
print('TB positive        :', int(d['y'].sum()))
print('previous TB        :', int(d['past_tb'].sum()))
print('fingerprint        :', d['fingerprint'][:16], '(matches pinned)')
print('concepts (exact)   :', len(names), names)
print('excluded           :', [c['concept'] for c in m['mapping']
                               if c['concept_idx'] is None])
print('images missing     :', len(missing))
print('images not approved:', len(extra), '(never read)')
assert not missing, 'approved images are missing: %s' % missing[:5]
print('expected similarity shape:', (len(d['pid']), len(names)))
print('\ncohort gate: PASSED')


## 4 - Inference

Fetches the pinned CLEAR assets (~1.73 GB on first use) and encodes the approved
images at 448 px. Only the 9 pinned rows of the concept tensor are used and the
tensor is memory-mapped, so its 1.13 GB is never materialised. The batch halves on
CUDA OOM. Shards are written as they are produced and keyed to the hash of the full
ordered id list, so re-running this cell resumes rather than restarts - and refuses
to resume onto a different id list.

**If the checkpoint's key layout cannot be matched to the encoder, this raises**
**`WeightLoadError` instead of encoding with a partly random network.**


In [ ]:
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
t0 = time.perf_counter()
S, prov = A.encode_concepts(d['pid'], device=dev, out_dir=WORK)
print('\nsimilarities', S.shape, ' %.1f min' % ((time.perf_counter()-t0)/60))
print(json.dumps(prov, indent=1, default=str))
assert S.shape == (len(d['pid']), len(names)), 'case-id alignment broken'
import numpy as np; assert np.isfinite(S).all()


## 5 - Diagnostic analysis

Four aggregate tables. Spearman rho between each concept similarity and the frozen
classifier score by group, with bootstrap CIs and **Benjamini-Hochberg** correction
across the 9 concepts; incremental R-squared over scanner, age, sex, symptom count
and previous TB; concept-score distributions by label and history; and each
concept's AUROC/AUPRC against the TB label, reported only where both classes have
at least 20 cases.

**Each concept score is a text similarity to one report sentence - a weakly**
**supervised proxy, not a validated detector and not radiologist ground truth.**
`past_tb` is clinical history, not an image annotation. Association is not
causation and does not show the classifier uses any visual cue.


In [ ]:
rows, incr = A.analyse(d, S, names)
dist = A.distributions(d, S, names)
disc = A.discrimination(d, S, names)
for nm, r in (('concept_associations', rows), ('concept_incremental_r2', incr),
              ('concept_distributions', dist), ('concept_discrimination', disc)):
    A._write(WORK/'tables'/(nm + '.csv'), r)
json.dump(A.run_manifest(d, p, prov), open(WORK/'run_manifest.json','w'),
          indent=1, default=str)
import pandas as pd
R = pd.DataFrame(rows)
print(R[R.group=='negatives (all)'][['concept','n','spearman_rho','ci95_lo',
      'ci95_hi','p_raw','p_bh_adjusted']].to_string(index=False))
print(); print(pd.DataFrame(incr).to_string(index=False))
print(); print(pd.DataFrame(disc).to_string(index=False))


## 6 - Validate the outputs, then download

Checks every expected file, column set and row count against
`EXPECTED_OUTPUTS.md`, and asserts that nothing per-case was written. Download the
files it lists from the notebook's Output tab. **Export aggregate tables only.**


In [ ]:
EXPECT = {'tables/concept_associations.csv':
            {'ci95_hi','ci95_lo','concept','group','n','p_bh_adjusted','p_raw',
             'spearman_rho'},
          'tables/concept_incremental_r2.csv':
            {'concept','delta_r2','r2_clinical_only','r2_with_concept'},
          'tables/concept_distributions.csv': None,
          'tables/concept_discrimination.csv': None,
          'tables/preflight_kaggle.json': None,
          'run_manifest.json': None}
import csv, io
bad = []
for rel, cols in EXPECT.items():
    f = WORK/rel
    if not f.is_file():
        bad.append('missing: ' + rel); continue
    if rel.endswith('.csv'):
        got = list(csv.DictReader(io.open(f, encoding='utf-8')))
        if cols and set(got[0]) != cols:
            bad.append('%s columns %s' % (rel, sorted(set(got[0]) ^ cols)))
        if len(got) > 200:
            bad.append('%s has %d rows - looks per-case' % (rel, len(got)))
    print('  %-42s %8d bytes' % (rel, f.stat().st_size))
assert not any(p.name.startswith('per_case') for p in WORK.rglob('*')), \
    'a per-case table was written; do not export it'
assert not bad, bad
print('\noutput gate: PASSED - download the files listed above')
print('\nReminders for whoever writes this up:')
for line in ('a similarity to the sentence "fibrosis" is NOT fibrosis detection',
             'TUNE is exposed; this is exploratory, never validation',
             'association is not causation; no claim that the classifier uses a cue',
             'cavity, nodule and reticular opacity were EXCLUDED, not measured',
             'the screening AUROC is unchanged at 0.7242'):
    print(' -', line)
